# Three kernel speedups you can watch on your own GPU

This notebook reproduces the three effects from *What is a GPU Kernel?* on whatever NVIDIA GPU you have:

1. A size that divides evenly by 64 lets the math library pick a faster kernel.
2. Merging two steps into one kernel halves the trips to the GPU's main memory.
3. A CUDA graph removes the gaps where the GPU waits for Python to start each small kernel.

In Colab: **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. It takes about two minutes.
Your numbers will differ from the issue's, which come from an A100: each GPU ships its own kernels and picks its own winners.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found. In Colab: Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0), "| PyTorch", torch.__version__)


def ms_per_call(fn, warmup=10, iters=50):
    """Average milliseconds per call, timed by the GPU itself so Python's own speed does not count."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / iters

## 1. A size that divides evenly by 64

nanoGPT's last layer turns each token's 768 numbers into one score per vocabulary entry, so it multiplies by a 768 x vocabulary matrix.
Karpathy padded the vocabulary from 50,257 entries to 50,304, a multiple of 64. Below, the same multiply runs at both sizes.
The padded one does 0.09 percent more work.

In [ ]:
tokens, width = 8 * 1024, 768          # 8 sequences of 1,024 tokens at nanoGPT's width
x = torch.randn(tokens, width, device="cuda", dtype=torch.float16)

times = {}
for vocab in (50257, 50304):
    w = torch.randn(width, vocab, device="cuda", dtype=torch.float16)
    times[vocab] = ms_per_call(lambda: x @ w)
    print(f"vocabulary {vocab:,}: {times[vocab]:.2f} ms per multiply")
    del w

print(f"the padded size runs {times[50257] / times[50304]:.2f}x as fast")

## 2. Merging two steps into one kernel

`x.cos().cos()` runs as two kernels in plain PyTorch. The first reads `x` from the GPU's main memory and writes the middle result back;
the second reads that result again and writes the final one. `torch.compile` merges both steps into one kernel that keeps the middle
result on the chip, so only two trips to main memory remain.

In [ ]:
x = torch.randn(64 * 1024 * 1024, device="cuda")   # 64 million numbers, 256 MB


def two_steps(t):
    return t.cos().cos()


merged = torch.compile(two_steps)
merged(x)                                          # the first call compiles the merged kernel; not timed

plain_ms = ms_per_call(lambda: two_steps(x))
merged_ms = ms_per_call(lambda: merged(x))
print(f"two kernels: {plain_ms:.2f} ms | one merged kernel: {merged_ms:.2f} ms | {plain_ms / merged_ms:.1f}x as fast")

## 3. A CUDA graph for many small kernels

A small model at batch size 1 runs one tiny kernel per layer. Python starts each one from the CPU, and the GPU often finishes
before the next one arrives, so it sits idle in the gaps. A CUDA graph records all the kernels once and replays them in one go.
This cell builds the graph by hand, the way PyTorch's CUDA graph docs do it; `torch.compile(model, mode="reduce-overhead")` sets one up for you.

In [ ]:
small = torch.nn.Sequential(*[torch.nn.Linear(256, 256) for _ in range(20)]).cuda().eval()
static_in = torch.randn(1, 256, device="cuda")

with torch.no_grad():
    side = torch.cuda.Stream()                    # warm up on a side stream before recording, as the docs require
    side.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(side):
        for _ in range(3):
            small(static_in)
    torch.cuda.current_stream().wait_stream(side)

    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        static_out = small(static_in)

    plain_ms = ms_per_call(lambda: small(static_in))
    graph_ms = ms_per_call(graph.replay)

print(f"20 layers, one kernel each: {plain_ms:.3f} ms | the same kernels replayed as a CUDA graph: {graph_ms:.3f} ms | {plain_ms / graph_ms:.1f}x as fast")

## Now check your own model

These three effects hide inside every training step. To find which one costs *your* model the most, record a trace of a few
training steps and run `python3 kernel_trace_check.py ./trace`. The README shows the profiler code and what the output means.